# SAMOSA Tutorial: Sampling from the Banana Distribution

This notebook demonstrates sampling strategies in the same order as the README: **Simple → Adaptation → Delayed rejection → Transport maps**. Each section runs a sampler and then shows scatter, trace, and lag plots.

In [ ]:
import numpy as np
from samosa import log_banana
from samosa import (
    load_samples,
    get_position_from_states,
    scatter_matrix,
    plot_trace,
    plot_lag,
)
from typing import Any, Dict
import matplotlib.pyplot as plt

burnin = 0.25

## Model Definition

All models in SAMOSA must satisfy the **ModelProtocol**: a callable that accepts `params` of shape `(d, 1)` and returns a dictionary. The model can include any number of additional keyword arguments so you can wrap a third party PDE solve to get its data. SAMOSA will always only call the model with params (position of the chain).

**Posterior specification (choose one):**
- **Option A:** Return `log_posterior` (float) directly
- **Option B:** Return both `log_prior` and `log_likelihood` (log_posterior = log_prior + log_likelihood)

**Optional keys:** `qoi`, `cost`, `model_output`, etc. — useful for MLMC and multi-fidelity methods.

The banana distribution uses a Rosenbrock-type transformation: `y0 = x0 - shift`, `y1 = x1 + y0^2`; the log PDF is multivariate normal in `y`.

In [ ]:
def banana_model(params: np.ndarray, cost: float = 2) -> Dict[str, Any]:
    """
    Banana model: log posterior via log_banana (Rosenbrock-type), optional qoi and cost.
    """
    # log_banana from samosa.utils.tools — transformation y0=x0, y1=x1+x0^2; then N(0,I) in y
    lp = log_banana(params)
    output = {
        "log_posterior": lp,
        "cost": cost * np.sum(params, axis=0),
        "qoi": np.sum(params, axis=0),
    }
    return output


# Alternative: component-based (log_prior + log_likelihood)
# def my_model(params):
#     return {"log_prior": ..., "log_likelihood": ...}

# Plot the banana_model contours to see what you're sampling from
xlims = [-3, 3]
ylims = [-6, 3]
x = np.linspace(xlims[0], xlims[1], 100)
y = np.linspace(ylims[0], ylims[1], 100)
X, Y = np.meshgrid(x, y)
points = np.vstack([X.ravel(), Y.ravel()])  # (2, N)
Z = np.exp(banana_model(points)["log_posterior"]).reshape(X.shape)
plt.contour(X, Y, Z, levels=10)
plt.show()

## Overview

See the README for **package hierarchy** (core, kernels, proposals, samplers, maps, utils).

We use several sampling strategies on the banana posterior. For each strategy we run the sampler and then show scatter, trace, and lag plots as diagnostics.

## 1. Base Gaussian random walk

In all the examples that follow, we first build a proposal, use that proposal to define a kernel, and use the kernel inside a sampler.

In [ ]:
from samosa import GaussianRandomWalk
from samosa import MetropolisHastingsKernel
from samosa import SingleChainSampler

from samosa import laplace_approx

# Laplace approximation for mean and covariance (MAP point + inverse Hessian)
map_point, cov_approx = laplace_approx(np.zeros((2, 1)), banana_model)
# Build a simple Gaussian random walk proposal using the MAP point and covariance
proposal = GaussianRandomWalk(mu=np.zeros((2, 1)), cov=cov_approx)
# Build a Metropolis-Hastings kernel using the model and proposal
kernel = MetropolisHastingsKernel(model=banana_model, proposal=proposal)
# Build a sampler using the kernel, initial position, number of iterations, and print/save intervals
sampler = SingleChainSampler(
    kernel=kernel,
    initial_position=map_point,
    n_iterations=10000,
    print_iteration=1000,
)
# To make sure you save your expensive sampler outputs, save to a directory
output = "banana-gaussian"
# Run the sampler
ar = sampler.run(output)
print(f"Acceptance rate: {ar:.4f}")

# Plots
# Load the samples from the sampler. Each sample is a ChainState object. See ChainState in samosa.core.state
samples = load_samples(output)
# Get the positions from the samples. Each position (location of the chain) is a numpy array of shape (d, 1)
positions = get_position_from_states(samples, burnin=burnin)
print(positions.shape)
# Scatter plot of the positions
fig, _, _ = scatter_matrix([positions])
plt.show()
# Trace plot of the positions
fig, _ = plot_trace(positions)
plt.show()
# Lag plot of the positions
fig, *_ = plot_lag(positions)
plt.show()

## 2. Adaptation (Haario)

Adapters are enhanced proposals that explore the posterior space more effectively. In this case, the Haario adapter improves the covariance of the proposal through empirical sample estimation.

In [ ]:
from samosa import HaarioAdapter
from samosa import AdaptiveProposal

map_point, cov_approx = laplace_approx(np.zeros((2, 1)), banana_model)
# Build a base proposal
base_proposal = GaussianRandomWalk(mu=np.zeros((2, 1)), cov=cov_approx)
# Build an adapter
adapter = HaarioAdapter(scale=2.38**2 / 2)
# Build an adaptive proposal (wraps the base proposal with an adapter, now acts as a proposal)
adaptive_proposal = AdaptiveProposal(base_proposal=base_proposal, adapter=adapter)
# Build a kernel using the model and adaptive proposal
kernel = MetropolisHastingsKernel(model=banana_model, proposal=adaptive_proposal)
sampler = SingleChainSampler(
    kernel=kernel,
    initial_position=map_point,
    n_iterations=10000,
    print_iteration=1000,
)
output_haario = "banana-haario"
ar = sampler.run(output_haario)
print(f"Acceptance rate: {ar:.4f}")

# Plots
samples = load_samples(output_haario)
positions = get_position_from_states(samples, burnin=burnin)
fig, _, _ = scatter_matrix([positions])
plt.show()
fig, _ = plot_trace(positions)
plt.show()
fig, *_ = plot_lag(positions)
plt.show()

## 3. Delayed rejection

This is another type of kernel that uses a two stage delayed mechanism to accept or reject proposed candidates. If you combine Haario's adaptation with this kernel, it is called DRAM - a popular state of the art sampler.

In [ ]:
from samosa import DelayedRejectionKernel

map_point, cov_approx = laplace_approx(np.zeros((2, 1)), banana_model)
base_proposal = GaussianRandomWalk(mu=np.zeros((2, 1)), cov=cov_approx)
adapter = HaarioAdapter(scale=2.38**2 / 2)
adaptive_proposal = AdaptiveProposal(base_proposal=base_proposal, adapter=adapter)
kernel = DelayedRejectionKernel(
    model=banana_model,
    proposal=adaptive_proposal,
    cov_scale=0.5,
)
sampler = SingleChainSampler(
    kernel=kernel,
    initial_position=map_point,
    n_iterations=10000,
    print_iteration=1000,
)
output_dr = "banana-delayedrejection"
ar = sampler.run(output_dr)
print(f"Acceptance rate: {ar:.4f}")

# Plots
samples = load_samples(output_dr)
positions = get_position_from_states(samples, burnin=burnin)
fig, _, _ = scatter_matrix([positions])
plt.show()
fig, _ = plot_trace(positions)
plt.show()
fig, *_ = plot_lag(positions)
plt.show()

## 4. Transport maps

Requires samples from a previous run (e.g. Haario) to adapt the map. `LowerTriangularMap` requires MParT.

We will use 2000 Haario samples to initially train the map and then continue sampling from it.
The mean and covariance of transport map based proposals must always be defined with respect to the reference space.
When restarting with previous samples, make sure you account for the iteration count of the samples being used. For example if using 2000 restarted samples, the iteration now starts from 2001, not 1.

In [ ]:
try:
    from samosa import TransportProposalBase
    from samosa import LowerTriangularMap

    # Load samples from Haario run; pass them to the map for initial adaptation
    samples_haario = load_samples(output_haario)
    n_adapt = min(2000, len(samples_haario))
    tmap = LowerTriangularMap(
        dim=2,
        total_order=2,
        adapt_start=500,
        adapt_end=10000,
        adapt_interval=500,
        initial_samples=samples_haario[:n_adapt],
    )
    base_proposal = GaussianRandomWalk(mu=np.zeros((2, 1)), cov=2.38**2 / 2 * np.eye(2))
    transport_proposal = TransportProposalBase(proposal=base_proposal, map=tmap)
    kernel = MetropolisHastingsKernel(model=banana_model, proposal=transport_proposal)
    sampler = SingleChainSampler(
        kernel=kernel,
        initial_position=samples_haario[-1].position,
        n_iterations=10000,
        print_iteration=1000,
    )
    output_transport = "banana-transport"
    ar = sampler.run(output_transport)
    print(f"Acceptance rate: {ar:.4f}")

    # Plots
    samples = load_samples(output_transport)
    positions = get_position_from_states(samples, burnin=0.5)
    fig, _, _ = scatter_matrix([positions])
    plt.show()
    fig, _ = plot_trace(positions)
    plt.show()
    fig, *_ = plot_lag(positions)
    plt.show()

    # Optional: reference positions in N(0,I) space
    from samosa.utils.post_processing import get_reference_position_from_states

    ref_positions = get_reference_position_from_states(
        load_samples(output_transport), 0.5
    )
    fig, _, _ = scatter_matrix(
        [ref_positions], sample_labels=[r"Reference positions ($r=T(x)$)"]
    )
    plt.show()
except ImportError as e:
    print(f"MParT not installed; skipping transport map example: {e}")

## 4.1 Transport map diagnostics

We can plot the pullback density of the transport map to visualize how well the map captures the target posterior.

In [ ]:
# Get the points making up the grid
x = np.vstack([X.ravel(), Y.ravel()])
pullback_pdf = tmap.pullback_logpdf(x)
# Plot the pull back density with the true density
plt.contour(X, Y, np.exp(pullback_pdf).reshape(X.shape), levels=10, linestyles="dashed")
plt.contour(X, Y, Z, levels=10)
plt.show()